# Startup Investment Outcome — Preprocessing Pipeline

Each step below implements a decision documented in `01_EDA.ipynb`.
The entire preprocessing is wrapped in a **sklearn Pipeline** that is fit on the training set only,
then applied identically to the validation and test sets.

In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv('data/investments_VC_clean.csv')
print(f"Loaded: {df.shape}")
df.head(3)

Loaded: (39802, 40)


,permalink,name,homepage_url,category_list,market,funding_total_usd,status,country_code,state_code,region,...,product_crowdfunding,round_A,round_B,round_C,round_D,round_E,round_F,round_G,round_H,market_category
0,/organization/waywire,#waywire,http://www.waywire.com,|Entertainment|Politics|Social Media|News|,News,1750000.0,acquired,USA,NY,New York City,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,content
1,/organization/tv-communications,&TV Communications,http://enjoyandtv.com,|Games|,Games,4000000.0,operating,USA,CA,Los Angeles,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,gaming
2,/organization/rock-your-paper,'Rock' Your Paper,http://www.rockyourpaper.org,|Publishing|Education|,Publishing,40000.0,operating,EST,NaN,Tallinn,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,content


## 1. Stratified Train / Val / Test Split (70 / 15 / 15)

In [3]:
# Encode target
status_map = {'closed': 0, 'operating': 1, 'acquired': 2}
df['status_code'] = df['status'].map(status_map)

X = df.drop(columns=['status_code'])
y = df['status_code']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15/0.85, stratify=y_train_val, random_state=42)

print(f"Train:      {X_train.shape[0]:,} rows")
print(f"Validation: {X_val.shape[0]:,} rows")
print(f"Test:       {X_test.shape[0]:,} rows")

for name, y_s in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    pct = y_s.value_counts(normalize=True).sort_index() * 100
    print(f"{name}: closed={pct.get(0,0):.1f}%  operating={pct.get(1,0):.1f}%  acquired={pct.get(2,0):.1f}%")

os.makedirs('data', exist_ok=True)
X_train.assign(status_code=y_train).to_csv('data/train_set.csv', index=False)
X_val.assign(status_code=y_val).to_csv('data/validation_set.csv', index=False)
X_test.assign(status_code=y_test).to_csv('data/test_set.csv', index=False)
print("\nRaw splits saved to data/")

Train:      27,860 rows
Validation: 5,971 rows
Test:       5,971 rows
Train: closed=5.4%  operating=86.5%  acquired=8.1%
Val: closed=5.4%  operating=86.5%  acquired=8.1%
Test: closed=5.4%  operating=86.5%  acquired=8.1%

Raw splits saved to data/


## 2. Custom Transformers

Each class implements one EDA decision. All fit on train only.

> **Note (pandas 2.x):** `pd.read_csv` infers text columns as `StringDtype`, which rejects numeric assignments.
> `DTypeConverter` runs first and converts all known numeric columns to `float64` by dropping and re-adding them.

In [4]:
# Step 0 — pandas 2.x StringDtype fix
# Drops and recreates numeric columns as float64 so all later transformers can assign freely.
class DTypeConverter(BaseEstimator, TransformerMixin):
    NUMERIC = [
        'funding_total_usd', 'funding_rounds',
        'seed', 'venture', 'angel', 'grant', 'private_equity', 'debt_financing',
        'equity_crowdfunding', 'convertible_note', 'undisclosed', 'product_crowdfunding',
        'post_ipo_equity', 'post_ipo_debt', 'secondary_market',
        'round_A', 'round_B', 'round_C', 'round_D', 'round_E',
        'round_F', 'round_G', 'round_H',
        'founded_year', 'founded_month', 'founded_quarter',
    ]
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        converted = {
            col: pd.to_numeric(X[col].astype(object), errors='coerce')
            for col in self.NUMERIC if col in X.columns
        }
        # Drop StringDtype columns, re-add as float64
        X = X.drop(columns=list(converted.keys()))
        for col, vals in converted.items():
            X[col] = vals
        return X

In [5]:
# EDA §3.2 — Fill founded_year/month/quarter nulls from founded_at
# (DTypeConverter already ensured these are float64, so simple fillna works)
class DateFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        at = pd.to_datetime(X['founded_at'], errors='coerce')
        X['founded_year']    = X['founded_year'].fillna(at.dt.year)
        X['founded_month']   = X['founded_month'].fillna(at.dt.month)
        X['founded_quarter'] = X['founded_quarter'].fillna(at.dt.quarter)
        return X

In [6]:
# EDA §3.3 — Fill null market with "Unknown" and map to market_category
# map_market_to_category is defined in 01_EDA.ipynb §3.3; reproduced here for pipeline self-containment.
_MARKET_CATEGORY_MAP = {}
_category_lists = {
    'admin_services': 'Employer Benefits Programs, Human Resource Automation, Corporate IT, Distribution, Service Providers, Archiving Service, Call Center, Collection Agency, College Recruiting, Courier Service, Debt Collections, Delivery, Document Preparation, Employee Benefits, Extermination Service, Facilities Support Services, Housekeeping Service, Human Resources, Knowledge Management, Office Administration, Packaging Services, Physical Security, Project Management, Staffing Agency, Trade Shows, Virtual Workforce',
    'advertising': 'Creative Industries, Promotional, Advertising Ad Exchange, Ad Network, Ad Retargeting, Ad Server, Ad Targeting, Advertising, Advertising Platforms, Affiliate Marketing, Local Advertising, Mobile Advertising, Outdoor Advertising, SEM, Social Media Advertising, Video Advertising',
    'agriculture': 'Agriculture, AgTech, Animal Feed, Aquaculture, Equestrian, Farming, Forestry, Horticulture, Hydroponics, Livestock',
    'app': 'Application Performance Monitoring, App Stores, Application Platforms, Enterprise Application, App Discovery, Apps, Consumer Applications, Enterprise Applications, Mobile Apps, Reading Apps, Web Apps',
    'artificial_intelligence': 'Artificial Intelligence, Intelligent Systems, Machine Learning, Natural Language Processing, Predictive Analytics',
    'biotechnology': 'Synthetic Biology, Bio-Pharm, Bioinformatics, Biometrics, Biopharma, Biotechnology, Genetics, Life Science, Neuroscience, Quantified Self',
    'clothing': 'Fashion, Laundry and Dry-cleaning, Lingerie, Shoes',
    'shopping': 'Consumer Behavior, Customer Support Tools, Discounts, Reviews and Recommendations, Auctions, Classifieds, Collectibles, Consumer Reviews, Coupons, E-Commerce, E-Commerce Platforms, Flash Sale, Gift, Gift Card, Gift Exchange, Gift Registry, Group Buying, Local Shopping, Made to Order, Marketplace, Online Auctions, Personalization, Point of Sale, Price Comparison, Rental, Retail, Retail Technology, Shopping, Shopping Mall, Social Shopping, Sporting Goods, Vending and Concessions, Virtual Goods, Wholesale',
    'community': "Self Development, Sex, Forums, Match-Making, Babies, Identity, Women, Kids, Entrepreneur, Networking, Adult, Baby, Cannabis, Children, Communities, Dating, Elderly, Family, Funerals, Humanitarian, Leisure, LGBT, Lifestyle, Men's, Online Forums, Parenting, Pet, Private Social Networking, Professional Networking, Q&A, Religion, Retirement, Sex Industry, Sex Tech, Social, Social Entrepreneurship, Teenagers, Virtual World, Wedding, Women's, Young Adults",
    'electronics': 'Mac, iPod Touch, Tablets, iPad, iPhone, Computer, Consumer Electronics, Drones, Electronics, Google Glass, Mobile Devices, Nintendo, Playstation, Roku, Smart Home, Wearables, Windows Phone, Xbox',
    'consumer_goods': 'Commodities, Sunglasses, Groceries, Batteries, Cars, Beauty, Comics, Consumer Goods, Cosmetics, DIY, Drones, Eyewear, Fast-Moving Consumer Goods, Flowers, Furniture, Green Consumer Goods, Handmade, Jewelry, Lingerie, Shoes, Tobacco, Toys',
    'content': 'E-Books, MicroBlogging, Opinions, Blogging Platforms, Content Delivery Network, Content Discovery, Content Syndication, Creative Agency, DRM, EBooks, Journalism, News, Photo Editing, Photo Sharing, Photography, Printing, Publishing, Social Bookmarking, Video Editing, Video Streaming',
    'data': 'Optimization, A/B Testing, Analytics, Application Performance Management, Artificial Intelligence, Big Data, Bioinformatics, Biometrics, Business Intelligence, Consumer Research, Data Integration, Data Mining, Data Visualization, Database, Facial Recognition, Geospatial, Image Recognition, Intelligent Systems, Location Based Services, Machine Learning, Market Research, Natural Language Processing, Predictive Analytics, Product Research, Quantified Self, Speech Recognition, Test and Measurement, Text Analytics, Usability Testing',
    'design': 'Visualization, Graphics, Design, Designers, CAD, Consumer Research, Data Visualization, Fashion, Graphic Design, Human Computer Interaction, Industrial Design, Interior Design, Market Research, Mechanical Design, Product Design, Product Research, Usability Testing, UX Design, Web Design',
    'education': 'Universities, College Campuses, University Students, High Schools, All Students, Colleges, Alumni, Charter Schools, College Recruiting, Continuing Education, Corporate Training, E-Learning, EdTech, Education, Edutainment, Higher Education, Language Learning, MOOC, Music Education, Personal Development, Primary Education, Secondary Education, Skill Assessment, STEM Education, Textbook, Training, Tutoring, Vocational Education',
    'energy': 'Gas, Natural Gas Uses, Oil, Oil & Gas, Battery, Biofuel, Biomass Energy, Clean Energy, Electrical Distribution, Energy, Energy Efficiency, Energy Management, Energy Storage, Fossil Fuels, Fuel, Fuel Cell, Oil and Gas, Power Grid, Renewable Energy, Solar, Wind Energy',
    'events': 'Concerts, Event Management, Event Promotion, Events, Nightclubs, Nightlife, Reservations, Ticketing, Wedding',
    'financial': 'Debt Collecting, P2P Money Transfer, Investment Management, Trading, Accounting, Angel Investment, Asset Management, Auto Insurance, Banking, Bitcoin, Commercial Insurance, Commercial Lending, Consumer Lending, Credit, Credit Bureau, Credit Cards, Crowdfunding, Cryptocurrency, Debit Cards, Debt Collections, Finance, Financial Exchanges, Financial Services, FinTech, Fraud Detection, Funding Platform, Gift Card, Health Insurance, Hedge Funds, Impact Investing, Incubators, Insurance, InsurTech, Leasing, Lending, Life Insurance, Micro Lending, Mobile Payments, Payments, Personal Finance, Prediction Markets, Property Insurance, Real Estate Investment, Stock Exchanges, Trading Platform, Transaction Processing, Venture Capital, Virtual Currency, Wealth Management',
    'food': 'Specialty Foods, Bakery, Brewing, Cannabis, Catering, Coffee, Confectionery, Cooking, Craft Beer, Dietary Supplements, Distillery, Farmers Market, Food and Beverage, Food Delivery, Food Processing, Food Trucks, Fruit, Grocery, Nutrition, Organic Food, Recipes, Restaurants, Seafood, Snack Food, Tea, Tobacco, Wine And Spirits, Winery',
    'gaming': 'Game, Games, Casual Games, Console Games, Contests, Fantasy Sports, Gambling, Gamification, Gaming, MMO Games, Online Games, PC Games, Serious Games, Video Games',
    'government': 'Polling, Governance, CivicTech, Government, GovTech, Law Enforcement, Military, National Security, Politics, Public Safety, Social Assistance',
    'hardware': 'Cable, 3D, 3D Technology, Application Specific Integrated Circuit (ASIC), Augmented Reality, Cloud Infrastructure, Communication Hardware, Communications Infrastructure, Computer, Computer Vision, Consumer Electronics, Data Center, Data Center Automation, Data Storage, Drone Management, Drones, DSP, Electronic Design Automation (EDA), Electronics, Embedded Systems, Field-Programmable Gate Array (FPGA), Flash Storage, Google Glass, GPS, GPU, Hardware, Industrial Design, Laser, Lighting, Mechanical Design, Mobile Devices, Network Hardware, NFC, Nintendo, Optical Communication, Playstation, Private Cloud, Retail Technology, RFID, RISC, Robotics, Roku, Satellite Communication, Semiconductor, Sensor, Sex Tech, Telecommunications, Video Conferencing, Virtual Reality, Virtualization, Wearables, Windows Phone, Wireless, Xbox',
    'health_care': 'Senior Health, Physicians, Electronic Health Records, Doctors, Healthcare Services, Diagnostics, Alternative Medicine, Assisted Living, Assistive Technology, Biopharma, Cannabis, Child Care, Clinical Trials, Cosmetic Surgery, Dental, Diabetes, Dietary Supplements, Elder Care, Electronic Health Record (EHR), Emergency Medicine, Employee Benefits, Fertility, First Aid, Funerals, Genetics, Health Care, Health Diagnostics, Home Health Care, Hospital, Medical, Medical Device, mHealth, Nursing and Residential Care, Nutraceutical, Nutrition, Outpatient Care, Personal Health, Pharmaceutical, Psychology, Rehabilitation, Therapeutics, Veterinary, Wellness',
    'it': 'Distributors, Algorithms, ICT, M2M, Technology, Business Information Systems, CivicTech, Cloud Data Services, Cloud Management, Cloud Security, CMS, Contact Management, CRM, Cyber Security, Data Center, Data Center Automation, Data Integration, Data Mining, Data Visualization, Document Management, E-Signature, Email, GovTech, Identity Management, Information and Communications Technology (ICT), Information Services, Information Technology, Intrusion Detection, IT Infrastructure, IT Management, Management Information Systems, Messaging, Military, Network Security, Penetration Testing, Private Cloud, Reputation, Sales Automation, Scheduling, Social CRM, Spam Filtering, Technical Support, Unified Communications, Video Chat, Video Conferencing, Virtualization, VoIP',
    'internet': 'Online Identity, Cyber, Portals, Web Presence Management, Domains, Tracking, Web Tools, Curated Web, Search, Cloud Computing, Cloud Data Services, Cloud Infrastructure, Cloud Management, Cloud Storage, Darknet, Domain Registrar, E-Commerce Platforms, Ediscovery, Email, Internet, Internet of Things, ISP, Location Based Services, Messaging, Music Streaming, Online Forums, Online Portals, Private Cloud, Product Search, Search Engine, SEM, Semantic Search, Semantic Web, SEO, SMS, Social Media, Social Media Management, Social Network, Unified Communications, Vertical Search, Video Chat, Video Conferencing, Visual Search, VoIP, Web Browsers, Web Hosting',
    'invest': 'Angel Investment, Banking, Commercial Lending, Consumer Lending, Credit, Credit Cards, Financial Exchanges, Funding Platform, Hedge Funds, Impact Investing, Incubators, Micro Lending, Stock Exchanges, Trading Platform, Venture Capital',
    'manufacturing': '3D Printing, Advanced Materials, Foundries, Industrial, Industrial Automation, Industrial Engineering, Industrial Manufacturing, Innovation Engineering, Civil Engineers, Heavy Industry, Engineering Firms, Systems, Machinery Manufacturing, Manufacturing, Paper Manufacturing, Plastics and Rubber Manufacturing, Textiles, Wood Processing',
    'media': 'Writers, Creative, Television, Entertainment, Media, Advice, Animation, Art, Audio, Audiobooks, Blogging Platforms, Broadcasting, Celebrity, Concerts, Content, Content Creators, Content Discovery, Content Syndication, Creative Agency, Digital Entertainment, Digital Media, DRM, EBooks, Edutainment, Event Management, Event Promotion, Events, Film, Film Distribution, Film Production, Guides, In-Flight Entertainment, Independent Music, Internet Radio, Journalism, Media and Entertainment, Motion Capture, Music, Music Education, Music Label, Music Streaming, Music Venues, Musical Instruments, News, Nightclubs, Nightlife, Performing Arts, Photo Editing, Photo Sharing, Photography, Podcast, Printing, Publishing, Reservations, Social Media, Social News, Theatre, Ticketing, TV, TV Production, Video, Video Editing, Video on Demand, Video Streaming, Virtual World',
    'messaging': 'Unifed Communications, Chat, Email, Meeting Software, Messaging, SMS, Unified Communications, Video Chat, Video Conferencing, VoIP, Wired Telecommunications',
    'mobile': 'Android, Google Glass, iOS, mHealth, Mobile, Mobile Apps, Mobile Devices, Mobile Payments, Windows Phone, Wireless',
    'music': 'Audio, Audiobooks, Independent Music, Internet Radio, Music, Music Education, Music Label, Music Streaming, Musical Instruments, Podcast',
    'natural_resources': 'Biofuel, Biomass Energy, Fossil Fuels, Mineral, Mining, Mining Technology, Natural Resources, Oil and Gas, Precious Metals, Solar, Timber, Water, Wind Energy',
    'navigation': 'Maps, Geospatial, GPS, Indoor Positioning, Location Based Services, Mapping Services, Navigation',
    'other': 'Mass Customization, Monetization, Testing, Subscription Businesses, Mobility, Incentives, Peer-to-Peer, Nonprofits, Alumni, Association, B2B, B2C, Blockchain, Charity, Collaboration, Collaborative Consumption, Commercial, Consumer, Crowdsourcing, Customer Service, Desktop Apps, Emerging Markets, Enterprise, Ethereum, Franchise, Freemium, Generation Y, Generation Z, Homeless Shelter, Infrastructure, Knowledge Management, LGBT Millennials, Non Profit, Peer to Peer, Professional Services, Project Management, Real Time, Retirement, Service Industry, Sharing Economy, Small and Medium Businesses, Social Bookmarking, Social Impact, Subscription Service, Technical Support, Underserved Children, Universities',
    'payment': 'Billing, Bitcoin, Credit Cards, Cryptocurrency, Debit Cards, Fraud Detection, Mobile Payments, Payments, Transaction Processing, Virtual Currency',
    'platforms': 'Development Platforms, Android, Facebook, Google, Google Glass, iOS, Linux, macOS, Nintendo, Operating Systems, Playstation, Roku, Tizen, Twitter, WebOS, Windows, Windows Phone, Xbox',
    'privacy': 'Digital Rights Management, Personal Data, Cloud Security, Corrections Facilities, Cyber Security, DRM, E-Signature, Fraud Detection, Homeland Security, Identity Management, Intrusion Detection, Law Enforcement, Network Security, Penetration Testing, Physical Security, Privacy, Security',
    'services': 'Funeral Industry, English-Speaking, Spas, Plumbers, Service Industries, Staffing Firms, Translation, Career Management, Business Services, Services, Accounting, Business Development, Career Planning, Compliance, Consulting, Customer Service, Employment, Environmental Consulting, Field Support, Freelance, Intellectual Property, Innovation Management, Legal, Legal Tech, Management Consulting, Outsourcing, Professional Networking, Quality Assurance, Recruiting, Risk Management, Social Recruiting, Translation Service',
    'real_estate': 'Office Space, Self Storage, Brokers, Storage, Home Owners, Realtors, Home & Garden, Utilities, Home Automation, Architecture, Building Maintenance, Building Material, Commercial Real Estate, Construction, Coworking, Facility Management, Fast-Moving Consumer Goods, Green Building, Home and Garden, Home Decor, Home Improvement, Home Renovation, Home Services, Interior Design, Janitorial Service, Landscaping, Property Development, Property Management, Real Estate, Real Estate Investment, Rental Property, Residential, Self-Storage, Smart Building, Smart Cities, Smart Home, Timeshare, Vacation Rental',
    'sales_marketing': 'Advertising, Affiliate Marketing, App Discovery, App Marketing, Brand Marketing, Cause Marketing, Content Marketing, CRM, Digital Marketing, Digital Signage, Direct Marketing, Direct Sales, Email Marketing, Lead Generation, Lead Management, Local, Local Advertising, Local Business, Loyalty Programs, Marketing, Marketing Automation, Mobile Advertising, Multi-level Marketing, Outdoor Advertising, Personal Branding, Public Relations, Sales, Sales Automation, SEM, SEO, Social CRM, Social Media Advertising, Social Media Management, Social Media Marketing, Sponsorship, Video Advertising',
    'science': 'Face Recognition, New Technologies, Advanced Materials, Aerospace, Artificial Intelligence, Bioinformatics, Biometrics, Biopharma, Biotechnology, Chemical, Chemical Engineering, Civil Engineering, Embedded Systems, Environmental Engineering, Human Computer Interaction, Industrial Automation, Industrial Engineering, Intelligent Systems, Laser, Life Science, Marine Technology, Mechanical Engineering, Nanotechnology, Neuroscience, Nuclear, Quantum Computing, Robotics, Semiconductor, Software Engineering, STEM Education',
    'software': 'Business Productivity, 3D Technology, Android, App Discovery, Application Performance Management, Apps, Artificial Intelligence, Augmented Reality, Billing, Bitcoin, Browser Extensions, CAD, Cloud Computing, Cloud Management, CMS, Computer Vision, Consumer Applications, Consumer Software, Contact Management, CRM, Cryptocurrency, Data Center Automation, Data Integration, Data Storage, Data Visualization, Database, Developer APIs, Developer Platform, Developer Tools, Document Management, Drone Management, E-Learning, EdTech, Electronic Design Automation (EDA), Embedded Software, Embedded Systems, Enterprise Applications, Enterprise Resource Planning (ERP), Enterprise Software, Facial Recognition, File Sharing, IaaS, Image Recognition, iOS, Linux, Machine Learning, macOS, Marketing Automation, Meeting Software, Mobile Apps, Mobile Payments, MOOC, Natural Language Processing, Open Source, Operating Systems, PaaS, Predictive Analytics, Presentation Software, Presentations, Private Cloud, Productivity Tools, QR Codes, Reading Apps, Retail Technology, Robotics, SaaS, Sales Automation, Scheduling, Sex Tech, Simulation, SNS, Social CRM, Software, Software Engineering, Speech Recognition, Task Management, Text Analytics, Transaction Processing, Video Conferencing, Virtual Assistant, Virtual Currency, Virtual Desktop, Virtual Goods, Virtual Reality, Virtual World, Virtualization, Web Apps, Web Browsers, Web Development',
    'sports': 'American Football, Baseball, Basketball, Boating, Cricket, Cycling, Diving, eSports, Fantasy Sports, Fitness, Golf, Hockey, Hunting, Outdoors, Racing, Recreation, Rugby, Sailing, Skiing, Soccer, Sporting Goods, Sports, Surfing, Swimming, Table Tennis, Tennis, Ultimate Frisbee, Volley Ball',
    'sustainability': 'Green, Wind, Biomass Power Generation, Renewable Tech, Environmental Innovation, Renewable Energies, Clean Technology, Biofuel, Biomass Energy, Clean Energy, CleanTech, Energy Efficiency, Environmental Engineering, Green Building, Green Consumer Goods, GreenTech, Natural Resources, Organic, Pollution Control, Recycling, Renewable Energy, Solar, Sustainability, Waste Management, Water Purification, Wind Energy',
    'transportation': 'Taxis, Air Transportation, Automotive, Autonomous Vehicles, Car Sharing, Courier Service, Delivery Service, Electric Vehicle, Ferry Service, Fleet Management, Food Delivery, Freight Service, Last Mile Transportation, Limousine Service, Logistics, Marine Transportation, Parking, Ports and Harbors, Procurement, Public Transportation, Railroad, Recreational Vehicles, Ride Sharing, Same Day Delivery, Shipping, Shipping Broker, Space Travel, Supply Chain Management, Taxi Service, Transportation, Warehousing, Water Transportation',
    'travel': 'Adventure Travel, Amusement Park and Arcade, Business Travel, Casino, Hospitality, Hotel, Museums and Historical Sites, Parks, Resorts, Timeshare, Tour Operator, Tourism, Travel, Travel Accommodations, Travel Agency, Vacation Rental',
    'video': 'Animation, Broadcasting, Film, Film Distribution, Film Production, Motion Capture, TV, TV Production, Video, Video Editing, Video on Demand, Video Streaming',
}
for _cat, _markets_str in _category_lists.items():
    for _m in _markets_str.split(', '):
        _m = _m.strip()
        if _m and _m not in _MARKET_CATEGORY_MAP:
            _MARKET_CATEGORY_MAP[_m] = _cat

def _map_market_to_category(market_value):
    if pd.isna(market_value):
        return 'Unknown'
    return _MARKET_CATEGORY_MAP.get(str(market_value).strip(), 'other')

class MarketFiller(BaseEstimator, TransformerMixin):
    """EDA §3.3 — Fill null market with 'Unknown' and derive market_category."""
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        X['market'] = X['market'].fillna('Unknown')
        if 'market_category' not in X.columns:
            X['market_category'] = X['market'].map(_map_market_to_category)
        return X


In [7]:
# EDA §3.4 — Fill country_code nulls with "Unknown"
# (analysis showed 0 rows where state_code could recover country_code)
class CountryFiller(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        X['country_code'] = X['country_code'].fillna('Unknown')
        return X

In [8]:
# EDA §3.4 — Create geo_cluster feature
class GeoClusterCreator(BaseEstimator, TransformerMixin):
    SV  = {'San Francisco','San Jose','Palo Alto','Mountain View','Menlo Park',
            'Redwood City','Sunnyvale','Santa Clara','Cupertino','San Mateo','Burlingame'}
    NY  = {'New York','Brooklyn','Manhattan','New York City'}
    BOS = {'Boston','Cambridge','Somerville','Waltham'}
    SEA = {'Seattle','Bellevue','Redmond','Kirkland'}
    LA  = {'Los Angeles','Santa Monica','Venice','Culver City','Pasadena'}

    def fit(self, X, y=None): return self
    def _cluster(self, row):
        cc, city = row.get('country_code',''), str(row.get('city',''))
        if pd.isna(cc) or cc == 'Unknown': return 'Unknown'
        if cc != 'USA': return cc
        if city in self.SV:  return 'USA-SiliconValley'
        if city in self.NY:  return 'USA-NY'
        if city in self.BOS: return 'USA-Boston'
        if city in self.SEA: return 'USA-Seattle'
        if city in self.LA:  return 'USA-LA'
        return 'USA-Other'
    def transform(self, X):
        X = X.copy()
        X['geo_cluster'] = X.apply(self._cluster, axis=1)
        return X

In [9]:
# EDA §4 — log1p transform on all funding amount columns
class FundingLogTransformer(BaseEstimator, TransformerMixin):
    COLS = ['funding_total_usd','seed','venture','angel','grant','private_equity',
            'debt_financing','equity_crowdfunding','convertible_note','undisclosed',
            'product_crowdfunding','post_ipo_equity','post_ipo_debt','secondary_market']
    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        for col in self.COLS:
            if col in X.columns:
                X[f'log_{col}'] = np.log1p(X[col].fillna(0))
        return X

In [10]:
# EDA §5 — Binary flags for rounds A/B/C; sum of D–H into round_D_plus; drop round_D through round_H
class RoundBinarizer(BaseEstimator, TransformerMixin):
    EARLY = ['round_A', 'round_B', 'round_C']
    LATE  = ['round_D', 'round_E', 'round_F', 'round_G', 'round_H']

    def fit(self, X, y=None): return self
    def transform(self, X):
        X = X.copy()
        for col in self.EARLY:
            if col in X.columns:
                X[f'has_{col}'] = (X[col].fillna(0) > 0).astype(int)
        late_cols = [c for c in self.LATE if c in X.columns]
        X['round_D_plus'] = X[late_cols].fillna(0).sum(axis=1)
        X = X.drop(columns=late_cols)
        return X

In [11]:
# EDA §6 — days_to_first_funding + median imputation
class FundingTimelineCalculator(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.median_ = None

    def fit(self, X, y=None):
        fa  = pd.to_datetime(X['founded_at'], errors='coerce')
        ffa = pd.to_datetime(X['first_funding_at'], errors='coerce')
        self.median_ = (ffa - fa).dt.days.median()
        return self

    def transform(self, X):
        X = X.copy()
        fa  = pd.to_datetime(X['founded_at'], errors='coerce')
        ffa = pd.to_datetime(X['first_funding_at'], errors='coerce')
        X['days_to_first_funding'] = (ffa - fa).dt.days.fillna(self.median_)
        return X

In [12]:
# EDA §7 — Target encoding for market_category, geo_cluster, and country_code
# Raw market is dropped after MarketFiller maps it to market_category.
class TargetEncoder(BaseEstimator, TransformerMixin):
    CATS = ['market_category', 'geo_cluster', 'country_code']

    def fit(self, X, y=None):
        self.encodings_ = {}
        self.global_means_ = {}
        if y is None: return self
        df_fit = X.copy()
        df_fit['_target'] = y.values if hasattr(y, 'values') else np.array(y)
        df_fit['_acquired'] = (df_fit['_target'] == 2).astype(int)
        df_fit['_closed']   = (df_fit['_target'] == 0).astype(int)
        for cat in self.CATS:
            if cat not in df_fit.columns: continue
            self.global_means_[cat] = {
                'acquired': df_fit['_acquired'].mean(),
                'closed':   df_fit['_closed'].mean()
            }
            self.encodings_[cat] = {
                'acquired': df_fit.groupby(cat)['_acquired'].mean(),
                'closed':   df_fit.groupby(cat)['_closed'].mean()
            }
        return self

    def transform(self, X):
        X = X.copy()
        for cat in self.CATS:
            if cat not in X.columns or cat not in self.encodings_: continue
            for outcome in ['acquired', 'closed']:
                mapping  = self.encodings_[cat][outcome]
                fallback = self.global_means_[cat][outcome]
                X[f'{cat}_enc_{outcome}'] = X[cat].map(mapping).fillna(fallback)
            X = X.drop(columns=[cat])
        return X


In [13]:
# EDA §3.2 — Median imputation for remaining founded_year/month/quarter nulls
# KNNImputer was too memory-intensive (~1 GB) at this data scale → replaced with per-column median
class DateMedianImputer(BaseEstimator, TransformerMixin):
    DATE_COLS = ['founded_year', 'founded_month', 'founded_quarter']

    def fit(self, X, y=None):
        self.medians_ = {col: X[col].median() for col in self.DATE_COLS if col in X.columns}
        return self

    def transform(self, X):
        X = X.copy()
        for col, med in self.medians_.items():
            if col in X.columns:
                X[col] = X[col].fillna(med)
        return X

In [14]:
# Drop columns not needed for modeling
class ColumnDropper(BaseEstimator, TransformerMixin):
    DROP = ['permalink', 'name', 'homepage_url', 'category_list',
            'status', 'state_code', 'region', 'city', 'status_label',
            'founded_at', 'first_funding_at', 'last_funding_at',
            'market',  # raw market replaced by market_category (encoded in TargetEncoder)
            ]
    def fit(self, X, y=None):
        self.fitted_ = True
        return self
    def transform(self, X):
        X = X.copy()
        to_drop = [c for c in self.DROP if c in X.columns]
        return X.drop(columns=to_drop)


## 3. Assemble and Run Pipeline

In [15]:
pipeline = Pipeline([
    ('dtype_converter',     DTypeConverter()),
    ('date_extractor',      DateFeatureExtractor()),
    ('country_filler',      CountryFiller()),
    ('market_filler',       MarketFiller()),
    ('geo_cluster',         GeoClusterCreator()),
    ('log_transformer',     FundingLogTransformer()),
    ('round_binarizer',     RoundBinarizer()),
    ('timeline_calculator', FundingTimelineCalculator()),
    ('target_encoder',      TargetEncoder()),
    ('date_median_imputer', DateMedianImputer()),
    ('column_dropper',      ColumnDropper()),
])

print("Pipeline steps:")
for name, step in pipeline.steps:
    print(f"  {name:25s} → {step.__class__.__name__}")

Pipeline steps:
  dtype_converter           → DTypeConverter
  date_extractor            → DateFeatureExtractor
  country_filler            → CountryFiller
  market_filler             → MarketFiller
  geo_cluster               → GeoClusterCreator
  log_transformer           → FundingLogTransformer
  round_binarizer           → RoundBinarizer
  timeline_calculator       → FundingTimelineCalculator
  target_encoder            → TargetEncoder
  date_median_imputer       → DateMedianImputer
  column_dropper            → ColumnDropper


In [16]:
pipeline.fit(X_train, y_train)
print("Pipeline fitted on training set.")

X_train_proc = pipeline.transform(X_train)
X_val_proc   = pipeline.transform(X_val)
X_test_proc  = pipeline.transform(X_test)
print(f"Train processed: {X_train_proc.shape}")
print(f"Val   processed: {X_val_proc.shape}")
print(f"Test  processed: {X_test_proc.shape}")

Pipeline fitted on training set.
Train processed: (27860, 46)
Val   processed: (5971, 46)
Test  processed: (5971, 46)


## 4. Verification

In [17]:
print("=== Shapes ===")
for name, X_p in [('Train', X_train_proc), ('Val', X_val_proc), ('Test', X_test_proc)]:
    print(f"{name}: {X_p.shape}")

=== Shapes ===
Train: (27860, 46)
Val: (5971, 46)
Test: (5971, 46)


In [18]:
print("=== Null counts after pipeline ===")
for name, X_p in [('Train', X_train_proc), ('Val', X_val_proc), ('Test', X_test_proc)]:
    nulls = X_p.isnull().sum().sum()
    print(f"{name}: {nulls} total nulls")

=== Null counts after pipeline ===
Train: 0 total nulls
Val: 0 total nulls
Test: 0 total nulls


In [19]:
# Which columns have nulls, and how many?
null_summary = X_train_proc.isnull().sum()
null_summary = null_summary[null_summary > 0].sort_values(ascending=False)
pct = (null_summary / len(X_train_proc) * 100).round(2)
print("Columns with nulls after pipeline (train):")
print(pd.DataFrame({'null_count': null_summary, 'null_pct': pct}).to_string())

Columns with nulls after pipeline (train):
Empty DataFrame
Columns: [null_count, null_pct]
Index: []


In [20]:
# Are the nulls concentrated in the same rows, or spread across different rows?
null_mask = X_train_proc.isnull().any(axis=1)
print(f"Rows with at least one null: {null_mask.sum():,} ({null_mask.mean()*100:.1f}%)")

# Show co-occurrence: which columns tend to be null together?
null_cols = X_train_proc.columns[X_train_proc.isnull().any()].tolist()
if null_cols:
    print(f"\nNull columns: {null_cols}")
    print("\nCo-occurrence matrix (how often two columns are null in the same row):")
    cooc = X_train_proc[null_cols].isnull().astype(int).T.dot(
           X_train_proc[null_cols].isnull().astype(int))
    print(cooc.to_string())
else:
    print("No nulls found — pipeline is clean!")

Rows with at least one null: 0 (0.0%)
No nulls found — pipeline is clean!


In [21]:
# For each null column: trace back which pipeline step should have handled it
# and show sample raw values to understand why it slipped through
if null_cols:
    for col in null_cols:
        null_rows = X_train_proc[X_train_proc[col].isnull()].index
        print(f"\n--- {col}: {len(null_rows)} nulls ---")
        # Show corresponding raw values from X_train
        if col in X_train.columns:
            print(f"  Raw values in X_train for null rows:")
            print(f"  {X_train.loc[null_rows, col].value_counts(dropna=False).head(5).to_dict()}")
        else:
            print(f"  (derived column — not in raw X_train)")
else:
    print("No nulls to trace.")

No nulls to trace.


In [22]:
print("=== Class distribution preserved? ===")
for name, y_s in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    pct = y_s.value_counts(normalize=True).sort_index() * 100
    print(f"{name}: closed={pct.get(0,0):.1f}%  operating={pct.get(1,0):.1f}%  acquired={pct.get(2,0):.1f}%")

=== Class distribution preserved? ===
Train: closed=5.4%  operating=86.5%  acquired=8.1%
Val: closed=5.4%  operating=86.5%  acquired=8.1%
Test: closed=5.4%  operating=86.5%  acquired=8.1%


In [23]:
print("=== Final feature list ===")
print(list(X_train_proc.columns))
print(f"\nTotal features: {X_train_proc.shape[1]}")

=== Final feature list ===
['funding_total_usd', 'funding_rounds', 'seed', 'venture', 'angel', 'grant', 'private_equity', 'debt_financing', 'equity_crowdfunding', 'convertible_note', 'undisclosed', 'product_crowdfunding', 'post_ipo_equity', 'post_ipo_debt', 'secondary_market', 'round_A', 'round_B', 'round_C', 'founded_year', 'founded_month', 'founded_quarter', 'log_funding_total_usd', 'log_seed', 'log_venture', 'log_angel', 'log_grant', 'log_private_equity', 'log_debt_financing', 'log_equity_crowdfunding', 'log_convertible_note', 'log_undisclosed', 'log_product_crowdfunding', 'log_post_ipo_equity', 'log_post_ipo_debt', 'log_secondary_market', 'has_round_A', 'has_round_B', 'has_round_C', 'round_D_plus', 'days_to_first_funding', 'market_category_enc_acquired', 'market_category_enc_closed', 'geo_cluster_enc_acquired', 'geo_cluster_enc_closed', 'country_code_enc_acquired', 'country_code_enc_closed']

Total features: 46


In [24]:
X_train_proc.assign(status_code=y_train.values).to_csv('data/train_processed.csv', index=False)
X_val_proc.assign(status_code=y_val.values).to_csv('data/val_processed.csv', index=False)
X_test_proc.assign(status_code=y_test.values).to_csv('data/test_processed.csv', index=False)
print("Saved: data/train_processed.csv, val_processed.csv, test_processed.csv")

Saved: data/train_processed.csv, val_processed.csv, test_processed.csv
